In [2]:
import pandas as pd
import numpy as np
from configs.settings import project_dir

In [91]:
p_dir = project_dir()

clean_data_dir = p_dir.CLEANED_DIR 

# need to clean orders, location, sellers and create translation_name table
orders = clean_data_dir/'cleaned_orders.csv'
sellers = clean_data_dir/'cleaned_sellers.csv'
location = clean_data_dir/'cleaned_location.csv'



In [13]:
orders_df = pd.read_csv(orders)
orders_df.sample(5)
orders_df = orders_df.drop('Unnamed: 0',axis=1)
orders_df.describe().T

,count,unique,top,freq
order_id,99441,99441,e481f51cbdc54678b7cc49136f2d6af7,1
customer_id,99441,99441,9ef432eb6251297304e76186b10a928d,1
order_status,99441,8,delivered,96478
order_purchase_timestamp,99441,98875,2018-03-31 15:08:21,3
order_approved_at,99281,90733,2018-02-27 04:31:10,9
order_delivered_carrier_date,97658,81018,2018-05-09 15:48:00,47
order_delivered_customer_date,96476,95664,2018-05-14 20:02:44,3
order_estimated_delivery_date,99441,459,2017-12-20 00:00:00,522
order_estimated_deliverey_date,99441,459,2017-12-20,522


In [25]:
print(orders_df.isnull().sum())
orders_df.groupby('order_status')[['order_approved_at','order_delivered_carrier_date','order_delivered_customer_date']].apply(
    lambda x : x.isnull().sum()
)


order_id                             0
customer_id                          0
order_status                         0
order_purchase_timestamp             0
order_approved_at                  160
order_delivered_carrier_date      1783
order_delivered_customer_date     2965
order_estimated_delivery_date        0
order_estimated_deliverey_date       0
dtype: int64


,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [84]:
orders_df.loc[
    (orders_df['order_status'] == 'approved') &
    (orders_df['order_approved_at'].notnull()) &
    (orders_df['order_delivered_carrier_date'].isnull()) &
    (orders_df['order_delivered_customer_date'].isnull())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_estimated_deliverey_date
44897,a2e4c44360b4a57bdff22f3a4630c173,8886130db0ea6e9e70ba0b03d7c0d286,approved,2017-02-06 20:18:17,2017-02-06 20:30:19,NaN,NaN,2017-03-01 00:00:00,2017-03-01
88457,132f1e724165a07f6362532bfb97486e,b2191912d8ad6eac2e4dc3b6e1459515,approved,2017-04-25 01:25:34,2017-04-30 20:32:41,NaN,NaN,2017-05-22 00:00:00,2017-05-22


In [ ]:
# handling the missing values
orders_df.groupby('order_status')['order_approved_at'].apply(
    lambda x : x.isnull().sum()
)
orders_df.loc[
    (orders_df['order_status'] == 'approved') & (orders_df['order_delivered_carrier_date'].isnull())]



,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_estimated_deliverey_date
44897,a2e4c44360b4a57bdff22f3a4630c173,8886130db0ea6e9e70ba0b03d7c0d286,approved,2017-02-06 20:18:17,2017-02-06 20:30:19,NaN,NaN,2017-03-01 00:00:00,2017-03-01
88457,132f1e724165a07f6362532bfb97486e,b2191912d8ad6eac2e4dc3b6e1459515,approved,2017-04-25 01:25:34,2017-04-30 20:32:41,NaN,NaN,2017-05-22 00:00:00,2017-05-22


In [85]:
orders_df.loc[
    (orders_df['order_status'] == 'delivered') &
    (orders_df['order_approved_at'].isnull())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_estimated_deliverey_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00,2017-03-17
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00,2017-03-22
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaN,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16 00:00:00,2017-03-16
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaN,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20 00:00:00,2017-03-20


In [202]:
location_df = pd.read_csv(location)
location_df.head(5)
location_df = location_df.drop('Unnamed: 0',axis=1)

In [150]:
location_df.head()

,zip_code,lat,lng,city,state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [182]:
location_df.groupby('city')[['zip_code','state']].value_counts().sample(10)

city              zip_code  state
goiânia           74320     GO        1
sao paulo         1105      SP       13
boa esperança     87390     PR        2
são paulo         5163      SP        1
centenario        77723     TO        3
jaguarari         48967     BA        2
ingaí             37215     MG        1
feira de santana  44009     BA       16
serrania          37143     MG       19
sao paulo         1502      SP        2
Name: count, dtype: int64

In [152]:
location_df[location_df['city'].str.contains('paulo')].sample(10)

,zip_code,lat,lng,city,state
3585,1331,-23.564900,-46.648305,sao paulo,SP
34780,3658,-23.529533,-46.497318,sao paulo,SP
121950,8473,-23.574919,-46.394990,sao paulo,SP
44447,4174,-23.633019,-46.607601,sao paulo,SP
51729,4425,-23.684323,-46.643084,sao paulo,SP
114230,8050,-23.513617,-46.452273,sao paulo,SP
13184,2361,-23.436312,-46.583004,são paulo,SP
32827,3570,-23.562830,-46.487892,sao paulo,SP
42161,4066,-23.611448,-46.650393,sao paulo,SP
26990,3266,-23.599361,-46.535726,sao paulo,SP


In [208]:
location_df['city_clean'] = location_df['city'].str.replace(
    {
        'ã' : 'a',
        'â' : 'a',
        'à' : 'a',
        'ä' : 'a',
        'å' : 'a',
        'á' : 'a',
        'ç' : 'c',
        'é' : 'e',
        'ü' : 'u',
        'ã' : 'a',
        'é' : 'e',
        'ê' : 'e',
        'ú' : 'u'
    }
)

In [194]:
location_df['city_clean'] = location_df['city'].str.replace(
    r"[^a-zA-Z\s]",'', regex=True
)

In [196]:
location_df[location_df['city'].str.contains('ã')][['city','city_clean']].value_counts()

city                   city_clean          
são paulo              so paulo                14521
são bernardo do campo  so bernardo do campo      897
ribeirão preto         ribeiro preto             590
são josé do rio preto  so jos do rio preto       495
são josé dos campos    so jos dos campos         474
                                               ...  
são domingos do sul    so domingos do sul          1
são valentim do sul    so valentim do sul          1
união da serra         unio da serra               1
são josé do herval     so jos do herval            1
são valentim           so valentim                 1
Name: count, Length: 586, dtype: int64

In [199]:
test = 'são joséê'
print(test.replace('ã', 'a'))

sao joséê


In [200]:
print(test.replace('ê', 'e'))

são josée


In [206]:
location_df['city_clean'] = location_df['city'].str.replace(
    'ã' , 'a'
)

In [211]:
location_df[
    location_df['city'].str.contains('ê')
][['city','city_clean']].sample(20)

,city,city_clean
271338,iepê,iepe
640134,xambrê,xambre
669185,xanxerê,xanxere
595000,três lagoas,tres lagoas
479512,santa inês,santa ines
617024,telêmaco borba,telemaco borba
641561,querência do norte,querencia do norte
694407,três coroas,tres coroas
324064,três rios,tres rios
431320,três corações,tres coracões


In [289]:
location_df = location_df.drop('city',axis=1)

In [290]:
location_df.to_csv(clean_data_dir/'cleaned_location.csv')

In [223]:
sellers_df = pd.read_csv(sellers)
sellers_df = sellers_df.drop('Unnamed: 0',axis=1)
sellers_df.sample(5)
sellers_df = sellers_df.rename(
    columns={
        'seller_zip_code_prefix' : 'zip_code'
    }
)

In [237]:
sellers_df[sellers_df['seller_city'].str.contains('ã')]

,seller_id,zip_code,seller_city,seller_state


In [270]:
sellers_df[sellers_df['seller_city'].str.contains('paulo')].sample(10)

,seller_id,zip_code,seller_city,seller_state
1698,2a7dc43cecabf23403078e2188437d1d,4142,sao paulo,SP
1122,8f0fbe2cd4d472157dc1cdef6edecaa9,2755,sao paulo,SP
2356,1bb2bdb95f4841f1bba2c0d2cd83d3c9,1257,sao paulo,SP
389,cea729054f157f5870bdd321a958d994,3161,sao paulo,SP
991,72c73be2b085b9d57650dd53eb2004c9,2116,sao paulo,SP
2839,c8660dcf9ba70575f45d80fe28c27713,2040,sao paulo,SP
246,71593c7413973a1e160057b80d4958f6,3407,sao paulo / sao paulo,SP
2914,808d4348b916efa08e766ebad39f61eb,2336,sao paulo,SP
1384,2b5ed0c9139dae8883a200dfcb272ece,4087,sao paulo,SP
2187,3be634553519fb6536a03e1358e9fdc7,8275,sao paulo,SP


In [271]:
sellers_df['city_clean'] = sellers_df['seller_city'].str.replace(
    {
        'ã' : 'a',
        'â' : 'a',
        'à' : 'a',
        'ä' : 'a',
        'å' : 'a',
        'á' : 'a',
        'ç' : 'c',
        'é' : 'e',
        'ü' : 'u',
        'ã' : 'a',
        'é' : 'e',
        'ê' : 'e',
        'ú' : 'u'
    }
)

In [275]:
sellers_df['city_clean'] = sellers_df['seller_city'].str.replace(
    r"[^a-zA-Z\s]", '', regex=True
)

In [272]:
sellers_df[['city_clean','seller_city']].sample(10)

,city_clean,seller_city
1813,sao paulo,sao paulo
2469,santa rosa de viterbo,santa rosa de viterbo
1459,parnamirim,parnamirim
135,sao paulo,sao paulo
668,sao paulo,sao paulo
1774,sao paulo,sao paulo
1201,canoas,canoas
1147,guarulhos,guarulhos
1904,sao paulo,sao paulo
931,osasco,osasco


In [288]:
sellers_df = sellers_df.drop('seller_city',axis=1)
sellers_df.head()

,seller_id,zip_code,seller_state,city_clean
0,3442f8959a84dea7ee197c632cb2df15,13023,SP,campinas
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,SP,mogi guacu
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,RJ,rio de janeiro
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,SP,sao paulo
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,SP,braganca paulista


In [291]:
sellers_df.to_csv(clean_data_dir/'cleaned_sellers.csv')

## category name translation table

In [278]:
category_name = p_dir.RAW_DATA_DIR/'product_category_name_translation.csv'
category_name

PosixPath('/home/arson/birdy/amit/noCartInsights/data/raw/product_category_name_translation.csv')

In [ ]:
category_name_df = pd.read_csv(category_name)
category_name_df.sample(5)


,product_category_name,product_category_name_english
44,industria_comercio_e_negocios,industry_commerce_and_business
53,pcs,computers
30,moveis_escritorio,office_furniture
38,moveis_colchao_e_estofado,furniture_mattress_and_upholstery
29,pet_shop,pet_shop


In [284]:
category_name_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB


In [285]:
category_name_df.duplicated().sum()

np.int64(0)

In [286]:
category_name_df.to_csv(clean_data_dir/'category_name.csv')